In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("IMDB dataset.csv")

In [3]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [4]:
# Removing HTML tags
import re 

def striphtml(data):
    p = re.compile(r'<.*?>')
    return p.sub('', data)

In [11]:
df['review'] = df['review'].apply(striphtml)

In [12]:
# Lowercase
df['review'] = df['review'].str.lower()


In [13]:
# Remove URLs
df['review'] = df['review'].str.replace(r'http\S+|www\.\S+', '', regex=True)


In [14]:
# Remove punctuation
df['review'] = df['review'].str.replace(r'[^a-zA-Z0-9\s]', ' ', regex=True)


In [15]:
# Remove extra spaces
df['review'] = df['review'].str.replace(r'\s+', ' ', regex=True).str.strip()


In [16]:
# Remove stopwords
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

stop = set(stopwords.words('english'))

df['review'] = df['review'].apply(
    lambda x: ' '.join([word for word in x.split() if word not in stop])
)



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\amank\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [17]:
df

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically family little boy jake thinks zombie...,negative
4,petter mattei love time money visually stunnin...,positive
...,...,...
49995,thought movie right good job creative original...,positive
49996,bad plot bad dialogue bad acting idiotic direc...,negative
49997,catholic taught parochial elementary schools n...,negative
49998,going disagree previous comment side maltin on...,negative


In [18]:
# Count total words in the entire corpus
total_words = df['review'].str.split().str.len().sum()

# Create a set of all unique words
vocab = set()

for text in df['review']:
    for word in text.split():
        vocab.add(word)

unique_words = len(vocab)

print("Total words in corpus:", total_words)
print("Vocabulary size:", unique_words)


Total words in corpus: 5978850
Vocabulary size: 103091


### One Hot Encoding:

In [20]:
df_ohe = pd.get_dummies(df, columns=['sentiment'])
df_ohe.head()


,review,sentiment_negative,sentiment_positive
0,one reviewers mentioned watching 1 oz episode ...,False,True
1,wonderful little production filming technique ...,False,True
2,thought wonderful way spend time hot summer we...,False,True
3,basically family little boy jake thinks zombie...,True,False
4,petter mattei love time money visually stunnin...,False,True


### Bag of Words:

In [23]:
from sklearn.feature_extraction.text import CountVectorizer
from collections import Counter

# Apply Bag of Words
cv = CountVectorizer()
X_bow = cv.fit_transform(df['review'])

# Vocabulary
vocab = cv.get_feature_names_out()
print("Vocabulary size:", len(vocab))

# Count occurrences
word_counts = X_bow.sum(axis=0).A1
vocab_counts = dict(zip(vocab, word_counts))

# Show most common 20 words
top20 = Counter(vocab_counts).most_common(20)
print("Top 20 most frequent words:\n", top20)


Vocabulary size: 103063
Top 20 most frequent words:
 [('movie', np.int64(87935)), ('film', np.int64(79671)), ('one', np.int64(53585)), ('like', np.int64(40160)), ('good', np.int64(29737)), ('time', np.int64(25098)), ('even', np.int64(24855)), ('would', np.int64(24599)), ('story', np.int64(23110)), ('really', np.int64(23090)), ('see', np.int64(23021)), ('well', np.int64(21260)), ('much', np.int64(19316)), ('bad', np.int64(18461)), ('get', np.int64(18414)), ('people', np.int64(18181)), ('great', np.int64(18137)), ('also', np.int64(17972)), ('first', np.int64(17576)), ('made', np.int64(16150))]


In [24]:

# assume df['review'] is preprocessed (lowercased, punctuation removed, etc.)
texts = df['review'].astype(str).tolist()   # ensure strings

def build_ngram_bow(texts, ngram_range=(2,2), min_df=1, max_features=None):
    """
    Build CountVectorizer with given ngram_range.
    min_df can be used to filter out extremely rare ngrams.
    """
    vec = CountVectorizer(ngram_range=ngram_range, min_df=min_df, max_features=max_features)
    X = vec.fit_transform(texts)
    vocab = vec.get_feature_names_out()
    counts = X.sum(axis=0).A1  # total counts per ngram across corpus
    # return sparse matrix, vocab array, counts array, vectorizer
    return X, vocab, counts, vec

# ---- bigrams ----
X_bi, vocab_bi, counts_bi, vec_bi = build_ngram_bow(texts, ngram_range=(2,2), min_df=2)  # min_df=2 removes ngrams appearing once
print("Bigrams shape:", X_bi.shape)
print("Bigrams vocabulary size:", len(vocab_bi))
print("Bigrams sparsity (%% non-zero): %.4f" % (100.0 * X_bi.nnz / (X_bi.shape[0] * X_bi.shape[1])))

# Show top 20 bigrams by corpus frequency
top20_bi_idx = np.argsort(counts_bi)[::-1][:20]
top20_bi = [(vocab_bi[i], int(counts_bi[i])) for i in top20_bi_idx]
print("Top 20 bigrams:", top20_bi)

# ---- trigrams ----
X_tri, vocab_tri, counts_tri, vec_tri = build_ngram_bow(texts, ngram_range=(3,3), min_df=2)
print("Trigrams shape:", X_tri.shape)
print("Trigrams vocabulary size:", len(vocab_tri))
print("Trigrams sparsity (%% non-zero): %.4f" % (100.0 * X_tri.nnz / (X_tri.shape[0] * X_tri.shape[1])))

# Show top 20 trigrams by corpus frequency
top20_tri_idx = np.argsort(counts_tri)[::-1][:20]
top20_tri = [(vocab_tri[i], int(counts_tri[i])) for i in top20_tri_idx]
print("Top 20 trigrams:", top20_tri)

# Optionally save vocab+counts to CSV
pd.DataFrame({"ngram": vocab_bi, "count": counts_bi}).sort_values("count", ascending=False).to_csv("bigrams_counts.csv", index=False)
pd.DataFrame({"ngram": vocab_tri, "count": counts_tri}).sort_values("count", ascending=False).to_csv("trigrams_counts.csv", index=False)
print("Saved bigrams_counts.csv and trigrams_counts.csv")


Bigrams shape: (50000, 613246)
Bigrams vocabulary size: 613246
Bigrams sparsity (% non-zero): 0.0107
Top 20 bigrams: [('ever seen', 2706), ('special effects', 2248), ('even though', 1999), ('one best', 1904), ('low budget', 1822), ('looks like', 1642), ('year old', 1598), ('waste time', 1505), ('see movie', 1500), ('much better', 1496), ('good movie', 1437), ('first time', 1415), ('sci fi', 1393), ('new york', 1315), ('watch movie', 1313), ('look like', 1273), ('years ago', 1256), ('real life', 1237), ('movie ever', 1232), ('high school', 1164)]
Trigrams shape: (50000, 217156)
Trigrams vocabulary size: 217156
Trigrams sparsity (% non-zero): 0.0057
Top 20 trigrams: [('movie ever seen', 525), ('worst movie ever', 469), ('movies ever seen', 380), ('one worst movies', 308), ('worst movies ever', 282), ('new york city', 277), ('world war ii', 231), ('films ever seen', 225), ('film ever seen', 222), ('movie ever made', 208), ('worst film ever', 206), ('waste time money', 182), ('based true s

# Observations About Dimensionality in Bag-of-Words, Bigrams, and Trigrams

## 1. Vocabulary Size Grows Rapidly With n
- **Unigram vocabulary** = number of unique individual words.
- **Bigram vocabulary** = unique contiguous 2-word sequences → usually **much larger** than unigram vocabulary.
- **Trigram vocabulary** = unique contiguous 3-word sequences → becomes **even larger**.
- **Reason:** As n increases, the number of possible combinations grows exponentially. Most sequences occur only once or a few times, causing vocabulary explosion.

---

## 2. Sparsity Increases
- Bag-of-Words matrices have shape **(n_documents × vocab_size)**.
- As `vocab_size` increases dramatically from unigrams → bigrams → trigrams:
  - Most cells become **zero** → extreme sparsity.
  - Sparse matrices help with memory, but computations become **slower** and **more costly**.

---

## 3. Many Rare n-grams
- Bigram and trigram vocabularies contain **many sequences that appear only once** (Zipf distribution).
- These very rare n-grams:
  - Increase noise.
  - Increase dimensionality.
  - Usually offer **little predictive value**.

---

## 4. Practical Consequences of Increasing n-gram Size
- **Higher memory usage:** Larger vocabularies require more RAM.
- **Higher computational cost:** Training ML models takes longer.
- **Overfitting risk increases:** Rare n-grams cause models to memorize noise.
- **Feature sparsity:** Many useless features dilute meaningful patterns.

---

## 5. Typical Mitigation Strategies
- **Set `min_df` (minimum document frequency)** to 2 or 3 to remove n-grams seen only once.
- **Limit `max_features`** to top-K features by frequency (e.g., 20k–50k).
- Use **TF-IDF** instead of raw counts to downweight rare words.
- Apply dimensionality reduction:
  - **SVD / TruncatedSVD**,  
  - **PCA** on dense vectors (after embeddings),  
  - **Autoencoders**.
- Use **embeddings** (Word2Vec, GloVe, BERT) rather than expanding to huge n-gram vocabularies.
- Combine **unigrams + bigrams** with bounded vocabulary size: `ngram_range=(1,2)` + `max_features`.

---

## 6. Empirical Notes (Typical for IMDB dataset)
These values vary depending on preprocessing:

- **Unigram vocabulary:** ~40k to 100k unique words.
- **Bigram vocabulary:** ~100k to 300k unique bigrams.
- **Trigram vocabulary:** 300k+ and can grow to millions if not constrained.
- Trigram features are mostly **rare**, leading to extremely sparse matrices.
- Cleaning choices (stopword removal, punctuation removal, lemmatization) significantly affect vocabulary size.

---

## Summary
Increasing n-gram size drastically increases vocabulary dimensionality, which:
- raises sparsity,
- increases memory and compute requirements,
- inflates noise,
- and often requires strong feature selection or dimensionality reduction.

For NLP tasks, combining **unigrams + bigrams** with controlled vocabulary size is typically the optimal balance.


# TF-IDF + Vocabulary + IDF Scores

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Apply TF-IDF
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(df['review'])

# Vocabulary (list of words)
vocab = tfidf.get_feature_names_out()

# IDF scores for each word
idf_scores = tfidf.idf_

# Combine into a DataFrame
idf_df = pd.DataFrame({
    "word": vocab,
    "idf_score": idf_scores
}).sort_values("idf_score", ascending=False)

print("Vocabulary size:", len(vocab))
print(idf_df.head(20))     # top 20 highest-idf (rarest) words


Vocabulary size: 103063
                                                word  idf_score
103062  zzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzz  11.126651
103061               zzzzzzzzzzzzzzzzzzzzzzzzzzzzzzz  11.126651
103060                            zzzzzzzzzzzzzzzzzz  11.126651
2                                        00000000000  11.126651
3                                      0000000000001  11.126651
103042                                        zwarts  11.126651
103041                                     zwartboek  11.126651
103039                                   zvyagvatsev  11.126651
103036                                        zvezda  11.126651
103035                                          zuzz  11.126651
103034                                        zutaut  11.126651
103033                                       zurlini  11.126651
103030                                       zuotian  11.126651
103029                                          zunz  11.126651
36              